<a href="https://colab.research.google.com/github/seferyetzirah/python4me/blob/master/zip2_congressional_dots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Ian Saltzman 11/10/26. Zip Codes of Students mapped to congressional districts -- represented by dots.

import pandas as pd
import folium
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Load dataset from Google Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/studentcongress.csv'
df = pd.read_csv(file_path)

# 3. Calculate unique student counts per district
district_summary = df.groupby('CongDistr')['State_StudentNumber'].nunique().to_dict()

# Extract key counts
pa02_count = district_summary.get('PA-02', 0) + district_summary.get('PA-02 (Split: PA-03)', 0)
pa03_count = district_summary.get('PA-03', 0) + district_summary.get('PA-03 (Split: PA-02)', 0) + district_summary.get('PA-03 (Split: PA-05)', 0)
pa05_count = district_summary.get('PA-05', 0) + district_summary.get('PA-05 (Split: PA-03)', 0)

# 4. Define your CARTO API Key and Map Tile URL
carto_api_key = "cb1_56bj_1_a22b9e74afce5ef22ae91e97"
tile_url = f"https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={carto_api_key}"

# 5. Initialize Folium Map centered on Philadelphia
m = folium.Map(
    location=[39.9800, -75.1400],
    zoom_start=11,
    tiles=tile_url,
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'
)

# 6. Define District Center Coordinates & Representative Data
districts_info = [
    {
        "district": "PA-02",
        "rep": "Brendan F. Boyle (D)",
        "lat": 40.0500,
        "lon": -75.0500,
        "color": "#2563eb",  # Blue
        "pure_count": district_summary.get('PA-02', 0),
        "split_count": district_summary.get('PA-02 (Split: PA-03)', 0),
        "total_count": pa02_count,
        "area": "Northeast Philly, Riverwards, Parts of North Philly"
    },
    {
        "district": "PA-03",
        "rep": "Dwight Evans (D)",
        "lat": 39.9700,
        "lon": -75.1900,
        "color": "#16a34a",  # Green
        "pure_count": district_summary.get('PA-03', 0),
        "split_count": district_summary.get('PA-03 (Split: PA-02)', 0) + district_summary.get('PA-03 (Split: PA-05)', 0),
        "total_count": pa03_count,
        "area": "West Philly, Center City, Northwest Philly, North Philly"
    },
    {
        "district": "PA-05",
        "rep": "Mary Gay Scanlon (D)",
        "lat": 39.9000,
        "lon": -75.1800,
        "color": "#dc2626",  # Red
        "pure_count": district_summary.get('PA-05', 0),
        "split_count": district_summary.get('PA-05 (Split: PA-03)', 0),
        "total_count": pa05_count,
        "area": "South Philly, Airport Region, Delaware County"
    }
]

# 7. Add Interactive District Markers & Popups to Map
for d in districts_info:
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; width: 220px;">
        <h3 style="margin-bottom: 2px; color: {d['color']};">{d['district']}</h3>
        <b>Representative:</b> {d['rep']}<br>
        <b>Primary Areas:</b> {d['area']}<hr style="margin: 6px 0;">
        <b>Total Students:</b> {d['total_count']}<br>
        <span style="font-size: 11px; color: #555;">
            - Pure District: {d['pure_count']}<br>
            - Split Boundary: {d['split_count']}
        </span>
    </div>
    """

    # Add Marker Circle
    folium.CircleMarker(
        location=[d['lat'], d['lon']],
        radius=14,
        popup=folium.Popup(popup_html, max_width=260),
        color=d['color'],
        fill=True,
        fill_color=d['color'],
        fill_opacity=0.7,
        tooltip=f"{d['district']}: {d['total_count']} Students"
    ).add_to(m)

# 8. Display the map directly in Colab
m




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
